In [7]:
import pandas as pd
import numpy as np
import requests
from bs4 import BeautifulSoup, Comment
import re
import time

## Todo
* minimize requests (combine injuries with https://www.basketball-reference.com/teams/%s/2024.html call)
* add features (historical_performances, fatigue, see more below...)
* optimize weights using some algorithm (gradient descent)
* automize picking (using voice assistant or making drag and drop)
* more advanced shit like context of player's role,  
* EDA of all features

In [21]:
conversion_table = {
    "ATL": "Atlanta Hawks",
    "BOS": "Boston Celtics",
    "CHO": "Charlotte Hornets",
    "CHI": "Chicago Bulls",
    "CLE": "Cleveland Cavaliers",
    "DAL": "Dallas Mavericks",
    "DEN": "Denver Nuggets",
    "DET": "Detroit Pistons",
    "GSW": "Golden State Warriors",
    "HOU": "Houston Rockets",
    "IND": "Indiana Pacers",
    "LAC": "Los Angeles Clippers",
    "LAL": "Los Angeles Lakers",
    "MEM": "Memphis Grizzlies",
    "MIA": "Miami Heat",
    "MIL": "Milwaukee Bucks",
    "MIN": "Minnesota Timberwolves",
    "NOP": "New Orleans Pelicans",
    "NYK": "New York Knicks",
    "BRK": "Brooklyn Nets",
    "OKC": "Oklahoma City Thunder",
    "ORL": "Orlando Magic",
    "PHI": "Philadelphia 76ers",
    "PHO": "Phoenix Suns",
    "POR": "Portland Trail Blazers",
    "SAC": "Sacramento Kings",
    "TOR": "Toronto Raptors",
    "UTA": "Utah Jazz",
    "WAS": "Washington Wizards",
    "SAS": "San Antonio Spurs",
}

In [22]:
def normalize(data):
    total = sum(data)
    normalized_data = [x / total for x in data]
    return normalized_data

### Version 1.2: using pure linear weights to calculate odds

Offensive/defensive ratings //
Injuries //
Historical performances //
Home court advantage //
Fatigue (recency of games) //
Possibly momentum? //
Sentiment analysis? //
Advanced stats?

With each of these features, assign a weight of importance to each feature, then pull the real time data and calculate the total score



In [23]:
weights = [
    0.4,
    0.4,
    0.2,
    None,
    None,
    None,
    None,
    None,
]  # offdef_ratings, injuries, homecourt_advantage, historical_performances, fatigue, momentum, sentiment, advanced

In [106]:
class Team:
    def __init__(self, team_abbr):
        self.offdef_ratings = self.find_offdef_ratings(team_abbr)
        self.team_abbr = team_abbr
        self.df = pd.read_html(
            "https://www.basketball-reference.com/teams/%s/2024.html" % team_abbr
        )
        self.sleep = time.sleep(4) # sleep between two calls 
        self.game_df = pd.read_html(
            "https://www.basketball-reference.com/teams/%s/2024_games.html" % team_abbr
        )
        self.injuries = self.find_injuries()
        self.player_scores = self.calculate_player_scores()
        self.team_score_with, self.team_score_without = self.calculate_team_score()
        self.momentum = self.calculate_momentum()

    def calculate_player_scores(self):
        """
        return (dict): player --> sum of weights
        """
        player_weights = [
            0.2,
            0.2,
            0.2,
            0.2,
            0.2,
        ]  # points, assists, rebounds, win shares, value over replacement player

        player_scores = {}

        for index, player in self.df[1].iterrows():  # pts, ast, rebs
            name = player["Player"]
            player_scores[name] = []
            player_scores[name].append(player["PTS"])
            player_scores[name].append(player["AST"])
            player_scores[name].append(player["TRB"])

        for index, player in self.df[3].iterrows():
            name = player["Player"]
            player_scores[name].append(player["BPM"])
            player_scores[name].append(player["VORP"])

        scores = []
        for player in player_scores:
            for i in range(len(player_scores[player])):
                player_scores[player][i] = player_scores[player][i] * player_weights[i]
            player_scores[player] = sum(player_scores[player])
            scores.append(player_scores[player])
        scores = normalize(scores)
        for i, player in enumerate(player_scores):
            player_scores[player] = scores[i]

        return player_scores

    def calculate_team_score(self):
        """
        players_scores (dict): player: value
        injuries (list) : [[out players], [day to day players]]
        return (float): team score with, team score without
        """

        team_score_with = 1  # consider day to day players as out
        team_score_without = 1  # don't consider day to day players as out

        for lst in self.injuries:
            for player in lst:
                if (
                    player not in self.player_scores
                ):  # for whatever reason, player doesnt show up on roster/injury report
                    pass
                else:
                    score = self.player_scores[player]
                    team_score_with -= score

        for player in self.injuries[0]:
            if player not in self.player_scores:
                pass
            else:
                score = self.player_scores[player]
                team_score_without -= score
        if len(self.injuries[1]) == 0:
            return team_score_with, team_score_without
        else:
            print(
                "Day to Day Players (check for most recent update): ", self.injuries[1]
            )
            return team_score_with, team_score_without

    def find_injuries(self):
        """
        return (dict): team: [[out players], [day to day players]]
        """
        time.sleep(4)
        df = pd.read_html("https://www.basketball-reference.com/friv/injuries.fcgi")
        injuries = {}
        for index, player in df[0].iterrows():
            if player["Team"] not in injuries:
                if "Day To Day" in player["Description"].split("-")[0]:
                    injuries[player["Team"]] = [[], [player["Player"]]]
                else:
                    injuries[player["Team"]] = [[player["Player"]], []]
            else:
                if "Day To Day" in player["Description"].split("-")[0]:
                    injuries[player["Team"]][1].append(player["Player"])
                else:  # player is out
                    injuries[player["Team"]][0].append(player["Player"])
        if conversion_table[self.team_abbr] in injuries:
            return injuries[conversion_table[self.team_abbr]]
        else:
            return [[], []]

    def find_offdef_ratings(self, team_abbr):
        """
        team_abbr (str) : team abbreivation (CHI, BOS, LAL)
        return (list[float]) : [off_rtg, def_rtg]
        """
        constant = 200  # for inverse relationship for defensive ratings
        url = "https://www.basketball-reference.com/teams/%s/2024.html" % team_abbr
        r = requests.get(url)
        soup = BeautifulSoup(r.text, "html.parser")
        tag = soup.find("div", {"id": "all_team_misc"})

        for element in tag(text=lambda text: isinstance(text, Comment)):
            soup = BeautifulSoup(element, "html.parser")

        rtgs = soup.find_all("td", {"data-stat": ["off_rtg", "def_rtg"]})
        rtgs = np.log2(
            [float(rtgs[0].get_text()), constant - float(rtgs[1].get_text())]
        )
        time.sleep(4)
        return rtgs

    def calculate_momentum(self):
        """
        return (float) : win percentage of last x number of games 
        """
        df = self.game_df[0]
        df = df[df["Unnamed: 7"].notna()]
        games_prior = 7 # number of games prior 
        last_games = df["Unnamed: 7"].tail(games_prior).tolist() 
        games_won = 0
        for game_result in last_games:
            if game_result == "W":
                games_won += 1
        win_percentage = games_won / games_prior
        
        return win_percentage

    def pipeline(self):
        """
        return (list): [offdef_ratings, injuries, historical_performances, fatigue, homecourt_advantage]
        """
        data = [
            sum(self.offdef_ratings),
            self.team_abbr,
            self.team_score_with,
            self.team_score_without,
            self.momentum
        ]

        return data

In [124]:
class Game:
    def __init__(self, home_team, away_team):
        self.home_team = home_team
        self.away_team = away_team

    def predict(self):
        """
        include_day_to_day (boolean) : decide whether to consider day to day players as out
        """
        home_team_sum_with = (
            sum(self.home_team.offdef_ratings)
            + self.home_team.team_score_with
            + 0.2
            + self.home_team.momentum
        )
        away_team_sum_with = (
            sum(self.away_team.offdef_ratings)
            + self.away_team.team_score_with
            + self.away_team.momentum
        )
        home_team_sum_without = (
            sum(self.home_team.offdef_ratings)
            + self.home_team.team_score_without
            + 0.2
            + self.home_team.momentum
        )
        away_team_sum_without = (
            sum(self.away_team.offdef_ratings)
            + self.away_team.team_score_without
            + self.home_team.momentum
        )

        if home_team_sum_with > away_team_sum_with:
            print("Winner with Day to Day Players out: " + self.home_team.team_abbr)
            print(
                self.home_team.team_abbr
                + ": "
                + str(home_team_sum_with)
                + " vs "
                + self.away_team.team_abbr
                + ": "
                + str(away_team_sum_with)
            )
        else:
            print("Winner with Day to Day Players out: " + self.away_team.team_abbr)
            print(
                self.home_team.team_abbr
                + ": "
                + str(home_team_sum_with)
                + " vs "
                + self.away_team.team_abbr
                + ": "
                + str(away_team_sum_with)
            )

        print("")

        if home_team_sum_without > away_team_sum_without:
            print("Winner with Day to Day Players out: " + self.home_team.team_abbr)
            print(
                self.home_team.team_abbr
                + ": "
                + str(home_team_sum_without)
                + " vs "
                + self.away_team.team_abbr
                + ": "
                + str(away_team_sum_without)
            )
        else:
            print("Winner with Day to Day Players out: " + self.away_team.team_abbr)
            print(
                self.home_team.team_abbr
                + ": "
                + str(home_team_sum_without)
                + " vs "
                + self.away_team.team_abbr
                + ": "
                + str(away_team_sum_without)
            )

### Outcome Testing

In [116]:
pelicans = Team("NOP")
wizards = Team("WAS")
Game(wizards, pelicans).predict()

Day to Day Players (check for most recent update):  ['Zion Williamson']
Day to Day Players (check for most recent update):  ['Daniel Gafford']
Winner with Day to Day Players out: NOP
WAS: 14.126280155901336 vs NOP: 14.582253541636561

Winner with Day to Day Players out: NOP
WAS: 14.234482599182313 vs NOP: 14.755925597063813


In [117]:
sixers = Team("PHI")
pistons = Team("DET")
Game(pistons, sixers).predict()

Day to Day Players (check for most recent update):  ['Marvin Bagley III', 'Jalen Duren']
Winner with Day to Day Players out: PHI
DET: 14.101695214687222 vs PHI: 15.083681309056443

Winner with Day to Day Players out: PHI
DET: 14.308949100697586 vs PHI: 15.083681309056443


In [119]:
hawks = Team("ATL")
raptors = Team("TOR")
Game(raptors, hawks).predict()

Day to Day Players (check for most recent update):  ["De'Andre Hunter"]
Day to Day Players (check for most recent update):  ['Chris Boucher', 'Otto Porter Jr.']
Winner with Day to Day Players out: TOR
TOR: 14.535436484847223 vs ATL: 14.330924901855658

Winner with Day to Day Players out: TOR
TOR: 14.576177225587964 vs ATL: 14.4563678501193


In [121]:
hornets = Team("CHO")
heat = Team("MIA")
Game(heat, hornets).predict()

Day to Day Players (check for most recent update):  ['Mark Williams']
Winner with Day to Day Players out: MIA
MIA: 14.639462237356707 vs CHO: 14.143604137715904

Winner with Day to Day Players out: MIA
MIA: 14.639462237356707 vs CHO: 14.242408918592398


In [125]:
lakers = Team("LAL")
spurs = Team("SAS")
Game(spurs, lakers).predict()

Winner with Day to Day Players out: LAL
SAS: 14.225688133793236 vs LAL: 14.85091537605983

Winner with Day to Day Players out: LAL
SAS: 14.225688133793236 vs LAL: 14.279486804631258


In [126]:
pacers = Team("IND")
bucks = Team("MIL")
Game(bucks, pacers).predict()

Winner with Day to Day Players out: MIL
MIL: 14.966174532398709 vs IND: 14.848703761474024

Winner with Day to Day Players out: MIL
MIL: 14.966174532398709 vs IND: 14.848703761474024


In [110]:
clippers = Team("LAC")
kings = Team("SAC")
Game(clippers, kings).predict()

Day to Day Players (check for most recent update):  ['Moussa Diabaté', 'Bones Hyland']
Winner with Day to Day Players out: LAC
LAC: 15.124778732075972 vs SAC: 14.906166057496147

Winner with Day to Day Players out: LAC
LAC: 15.188665156210844 vs SAC: 14.906166057496147


In [113]:
lakers = Team("LAL")
mavericks = Team("DAL")
Game(mavericks, lakers).predict()

Day to Day Players (check for most recent update):  ['Anthony Davis', 'LeBron James', 'Jarred Vanderbilt']
Day to Day Players (check for most recent update):  ['Luka Dončić', 'Tim Hardaway Jr.', 'Derrick Jones Jr.']
Winner with Day to Day Players out: LAL
DAL: 14.311299251105408 vs LAL: 14.395470501804091

Winner with Day to Day Players out: LAL
DAL: 14.79874158836835 vs LAL: 15.030682661398771


In [114]:
bulls = Team("CHI")
nuggets = Team("DEN")
Game(nuggets, bulls).predict()

Day to Day Players (check for most recent update):  ['Alex Caruso', 'Torrey Craig', 'Patrick Williams']
Day to Day Players (check for most recent update):  ['Aaron Gordon', 'Jamal Murray']
Winner with Day to Day Players out: DEN
DEN: 14.822222304193621 vs CHI: 14.315932418541367

Winner with Day to Day Players out: DEN
DEN: 15.070967169551905 vs CHI: 14.57909031327821


In [115]:
celtics = Team("BOS")
cavs = Team("CLE")
Game(celtics, cavs).predict()

Day to Day Players (check for most recent update):  ['Caris LeVert']
Winner with Day to Day Players out: BOS
BOS: 15.295647506106159 vs CLE: 14.754125793176152

Winner with Day to Day Players out: BOS
BOS: 15.295647506106159 vs CLE: 14.847502058125555


### Code Testing

In [89]:
df = pd.read_html("https://www.basketball-reference.com/teams/SAC/2024_games.html")

In [95]:
df = df[0]

In [96]:
df

,G,Date,Start (ET),Unnamed: 3,Unnamed: 4,Unnamed: 5,Opponent,Unnamed: 7,Unnamed: 8,Tm,Opp,W,L,Streak,Notes
0,1,"Wed, Oct 25, 2023",9:00p,NaN,Box Score,@,Utah Jazz,W,NaN,130,114,1,0,W 1,NaN
1,2,"Fri, Oct 27, 2023",10:00p,NaN,Box Score,NaN,Golden State Warriors,L,NaN,114,122,1,1,L 1,NaN
2,3,"Sun, Oct 29, 2023",9:00p,NaN,Box Score,NaN,Los Angeles Lakers,W,OT,132,127,2,1,W 1,NaN
3,4,"Wed, Nov 1, 2023",10:00p,NaN,Box Score,@,Golden State Warriors,L,NaN,101,102,2,2,L 1,NaN
4,5,"Sat, Nov 4, 2023",8:00p,NaN,Box Score,@,Houston Rockets,L,NaN,89,107,2,3,L 2,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
81,79,"Tue, Apr 9, 2024",8:00p,NaN,NaN,@,Oklahoma City Thunder,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
82,80,"Thu, Apr 11, 2024",10:00p,NaN,NaN,NaN,New Orleans Pelicans,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
83,G,Date,Start (ET),NaN,NaN,NaN,Opponent,NaN,NaN,Tm,Opp,W,L,Streak,Notes
84,81,"Fri, Apr 12, 2024",10:30p,NaN,NaN,NaN,Phoenix Suns,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [97]:
df = df[df["Unnamed: 7"].notna()]

In [102]:
df["Unnamed: 7"].tail(7).tolist()

['W', 'W', 'L', 'W', 'L', 'W', 'W']

In [ ]:
df = pd.read_html("https://www.basketball-reference.com/friv/injuries.fcgi")

## Find players that are out (cbssports method)

In [ ]:
# def find_team_name(tag):
#     pattern = r'>(.*?)<\/a>'
#     team_name = str(tag.find_all(class_ = "TeamName")[0])
#     team_name = re.search(pattern, team_name).group(1)
#     team_name = team_name.split('>', 1)[-1]
#     return team_name

In [ ]:
# def find_injury_time(tag):
#     element = str(tag).split("width: 40%;")[1]
#     start_index = element.find('">') + 2
#     end_index = element.find('</td>')
#     injury_time = element[start_index:end_index].strip()
#     if injury_time != "Game Time Decision":
#         return "Out"
#     else:
#         return "Questionable"

In [ ]:
# def find_injuries(teams_element):
#     injuries = {}
#     for team_element in teams_element:
#         team_name = find_team_name(team_element)
#         injuries[team_name] = []
#         for report_element in team_element.find_all('tr', class_='TableBase-bodyTr'):
#             player_name = report_element.find('span', class_='CellPlayerName--long').text
#             # injury_time = find_injury_time(report_element)
#             injuries[team_name].append(player_name)
#     return injuries


In [ ]:
# r = requests.get('https://www.cbssports.com/nba/injuries/')
# soup = BeautifulSoup(r.text, 'html.parser')
# teams = soup.find_all(class_ = 'TableBaseWrapper')

# injuries = find_injuries(teams)